# Kandinsky V2A — Six-Scene Listening Pilot

This notebook downloads **six short CC0 videos automatically**, adds one fixed sound-event prompt to each, generates **one five-second sound per video** with Kandinsky Lite Distill, and creates a blinded listener package.

You do not need to upload any videos. The notebook fetches them from Wikimedia Commons and records their source pages and CC0 status. The generated clips are muted video windows; their original audio is discarded.

**Pilot question:** across different visible events, how well do Kandinsky's generated sounds fit the scene, line up in time, and hold up in quality?

This is not model training, a seed comparison, or a broad success-rate benchmark. The same seed (42), checkpoint, duration, and generation settings are used for every scene; only the video and its short event prompt change.

## Run order
1. Select an A100 GPU runtime and run the cells in order.
2. The notebook downloads and prepares six CC0 clips automatically.
3. Check the built-in prompts in the scene table; they are concise sound descriptions, not visual-story captions.
4. Run one Kandinsky generation for each clip.
5. Download the final researcher ZIP. Share only Participant_Package.zip with raters; keep researcher_key.json private.


## 1 — Check the Colab runtime

The working Kandinsky setup uses Python 3.13 or newer and the A100 profile. The cell stops with a clear message if Colab is attached to a CPU runtime or a different GPU.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, urllib.request
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote

print("Python:", sys.version)
if sys.version_info < (3, 13):
    raise RuntimeError("This Kandinsky setup expects Python 3.13 or newer.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. In Colab, choose Runtime > Change runtime type > A100 GPU.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("Could not inspect the GPU: " + gpu.stderr[-1000:])
print("GPU:", gpu.stdout.strip())
if "A100" not in gpu.stdout:
    print("The notebook was prepared around the A100 profile used in the successful Kandinsky run.")

WORK = Path("/content/kandinsky_multivideo_review")
INPUT_DIR = WORK / "original_sources"
WINDOW_DIR = WORK / "scene_windows"
GEN_DIR = WORK / "generated"
BUNDLE = WORK / "kandinsky_review_bundle.zip"
for folder in (INPUT_DIR, WINDOW_DIR, GEN_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", WORK)


## 2 — Install Kandinsky Lite Distill

This is the same official Kandinsky repository, checkpoint, and SDPA attention route used by the working single-video notebook. This notebook does not request API keys.

In [ ]:
REPO = Path("/content/kandinsky-6")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
if not Path("/usr/local/bin/just").exists():
    subprocess.run(["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"], check=True)
subprocess.run(["just", "setup"], cwd=REPO, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Kandinsky source commit:", SOURCE_COMMIT)
subprocess.run(["just", "download", "lite-distill"], cwd=REPO, check=True)
WEIGHT_REPO_CACHE = Path.home() / ".cache/kandinsky/weights/models--kandinskylab--Kandinsky-6.0-Lite-distill-5s-Diffusers"
CHECKPOINT_REF = WEIGHT_REPO_CACHE / "refs/main"
CHECKPOINT_REVISION = CHECKPOINT_REF.read_text(encoding="utf-8").strip() if CHECKPOINT_REF.is_file() else "not_detected"
print("Lite Distill checkpoint revision:", CHECKPOINT_REVISION)
print("Model is ready.")


## 3 — Download six CC0 clips and prepare five-second windows

The source list is fixed and documented below. Each Wikimedia Commons file page labels its video CC0 1.0. This code downloads the source files directly, checks each duration, strips its original sound, and converts the selected five-second section to MP4.

The two bird clips intentionally give us related but different visuals; the other clips cover city walking, running water, a fireplace, and a door opening. This lets us see whether the system works beyond a single event family.

In [ ]:
SCENES = [
    {
        "scene_id": "city_walkers", "label": "City pedestrians walking",
        "filename": "Big City Life.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Big_City_Life.webm",
        "source_license": "CC0 1.0",
        "prompt": "Several pedestrians walking on a city pavement, with soft footsteps and distant urban movement. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "bird_on_ledge", "label": "Small bird walking on a ledge",
        "filename": "Cute Colorful Bird on Walking on Ledge.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Cute_Colorful_Bird_on_Walking_on_Ledge.webm",
        "source_license": "CC0 1.0",
        "prompt": "A small bird hops and walks along a ledge, with a few light nearby chirps. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "curious_bird", "label": "Curious bird looking at a camera",
        "filename": "Curious Little Bird Looking at a Camera.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Curious_Little_Bird_Looking_at_a_Camera.webm",
        "source_license": "CC0 1.0",
        "prompt": "A small songbird moves nearby and gives a few clear, natural chirps in a quiet outdoor setting. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "running_stream", "label": "Water flowing through a stream",
        "filename": "Water_flowing_strem.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:Water_flowing_strem.webm",
        "source_license": "CC0 1.0",
        "prompt": "Steady natural water flows over a shallow stream bed, with gentle continuous rippling. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "fireplace", "label": "Fire burning in a fireplace",
        "filename": "ABOedit2015-Video-Fire.webm",
        "source_page": "https://commons.wikimedia.org/wiki/File:ABOedit2015-Video-Fire.webm",
        "source_license": "CC0 1.0",
        "prompt": "A small indoor fireplace burns with a gentle crackle and quiet room tone. No music or speech.",
        "start_seconds": 0,
    },
    {
        "scene_id": "door_opening", "label": "Door opening",
        "filename": "DoorOpen.gif",
        "source_page": "https://commons.wikimedia.org/wiki/File:DoorOpen.gif",
        "source_license": "CC0 1.0",
        "prompt": "A door slowly opens with a short creaking hinge and a subtle latch movement. No music or speech.",
        "start_seconds": 0,
    },
]

GENERATION_SECONDS = 5
SEED = 42
USER_AGENT = "Kandinsky-V2A-Listening-Pilot/1.0 (research download)"
SOURCE_METADATA = []
SCENE_WINDOWS = []

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def run_command(command, cwd=None, print_tail=1800):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-print_tail:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-4500:])
        raise RuntimeError("Command failed. See the output above.")
    return result

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("ffmpeg and ffprobe are required; they are normally available in Colab.")

for item in SCENES:
    encoded_name = quote(item["filename"], safe="")
    url = "https://commons.wikimedia.org/wiki/Special:Redirect/file/" + encoded_name
    source_path = INPUT_DIR / item["filename"]
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=180) as response:
        payload = response.read()
        final_url = response.geturl()
    if len(payload) < 10_000:
        raise RuntimeError("Download looks incomplete for " + item["filename"])
    source_path.write_bytes(payload)
    probe = run_command(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                         "-of", "default=noprint_wrappers=1:nokey=1", str(source_path)])
    source_duration = float(probe.stdout.strip())
    start = float(item["start_seconds"])
    if start + GENERATION_SECONDS > source_duration:
        raise ValueError(item["filename"] + " is too short for the selected five-second window.")
    scene_video = WINDOW_DIR / (item["scene_id"] + ".mp4")
    run_command(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(source_path),
                 "-ss", str(start), "-t", str(GENERATION_SECONDS),
                 "-map", "0:v:0", "-an", "-vf", "fps=24,scale='min(960,iw)':-2",
                 "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
                 "-pix_fmt", "yuv420p", str(scene_video)])
    item["source_path_local"] = source_path
    item["scene_video_local"] = scene_video
    SOURCE_METADATA.append({
        **{k: item[k] for k in ("scene_id", "label", "filename", "source_page", "source_license", "prompt", "start_seconds")},
        "download_url": url, "resolved_download_url": final_url,
        "source_sha256": sha256_file(source_path), "source_duration_seconds": source_duration,
    })
    SCENE_WINDOWS.append({
        "scene_id": item["scene_id"], "source_name": item["filename"],
        "source_sha256": sha256_file(source_path), "source_page": item["source_page"],
        "source_license": item["source_license"], "start_seconds": start,
        "event_prompt": item["prompt"], "video_path": "videos/" + item["scene_id"] + ".mp4",
        "video_sha256": sha256_file(scene_video), "duration_seconds": GENERATION_SECONDS,
    })
    print("Ready:", item["label"], "|", round(source_duration, 2), "s source |", scene_video)

print("Prepared", len(SCENE_WINDOWS), "source scenes.")
print("Prompts used for the study:")
for item in SCENES:
    print("-", item["label"] + ":", item["prompt"])


## 4 — Generate one sound for each video

Every scene uses the same fixed seed (42), Lite Distill checkpoint, SDPA attention, five-second duration, and disabled prompt expansion. The scene video and its short event prompt are the only per-scene inputs.

The generations run sequentially, one at a time. If one scene fails, the notebook records that failure and continues; it can still create the listening package if at least five scenes succeed.

In [ ]:
DEVICE_CONFIG = REPO / "kandinsky/configs/devices/a100-lite-distill.yaml"
DEVICE_CONFIG.write_text(
    "checkpoint: lite-distill\ncache:\n  mode: none\noffload:\n  strategy: module\n"
    "compile:\n  strategy: none\nattention:\n  engine: sdpa\n"
    "text_embedder:\n  quantized_qwen: false\nbeautifier:\n  name: none\n",
    encoding="utf-8")

successful_scenes = []
candidates = []
generation_failures = []

for scene_number, (item, scene_meta) in enumerate(zip(SCENES, SCENE_WINDOWS), start=1):
    scene_id = item["scene_id"]
    scene_video = item["scene_video_local"]
    candidate_id = "audio_" + scene_id
    output_mp4 = GEN_DIR / (candidate_id + ".mp4")
    output_wav = GEN_DIR / (candidate_id + ".wav")
    command = ["just", "generate", item["prompt"], "--config", str(DEVICE_CONFIG),
               "--attention-engine", "sdpa", "--video-file", str(scene_video),
               "--time-length", str(GENERATION_SECONDS), "--out", str(output_mp4),
               "--seed", str(SEED)]
    print("\nGenerating scene", scene_number, "of", len(SCENES), ":", item["label"])
    print("Prompt:", item["prompt"])
    started = time.time()
    try:
        run_command(command, cwd=REPO, print_tail=2200)
        if not output_mp4.is_file():
            raise FileNotFoundError("Kandinsky output MP4 was not created.")
        run_command(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(output_mp4),
                     "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(output_wav)])
        elapsed = round(time.time() - started, 2)
        successful_scenes.append(scene_meta)
        candidates.append({
            "candidate_id": candidate_id, "scene_id": scene_id, "seed": SEED,
            "event_prompt": item["prompt"], "audio_path": "audio/" + candidate_id + ".wav",
            "audio_sha256": sha256_file(output_wav),
            "generated_mp4_path": "generated/" + candidate_id + ".mp4",
            "output_mp4_sha256": sha256_file(output_mp4),
            "generation_seconds": elapsed,
        })
        print("Finished in", elapsed, "seconds.")
    except Exception as error:
        generation_failures.append({"scene_id": scene_id, "label": item["label"], "error": str(error)})
        print("FAILED:", item["label"], "—", str(error))

print("\nSuccessful scenes:", len(successful_scenes), "/", len(SCENES))
if generation_failures:
    print("Generation failures:", json.dumps(generation_failures, ensure_ascii=False, indent=2))
if len(successful_scenes) < 5:
    raise RuntimeError("Fewer than five scenes generated; no rating package was built.")


## 5 — Create the listener ZIP and private researcher files

The participant ZIP contains one muted video and one generated WAV for each successful scene. The prompt and seed are kept out of the rating page. Scene order is randomized for each rater; no candidate or seed comparison is shown.

The complete artifact download contains both the participant ZIP and the private key, so share only Participant_Package.zip with raters.

In [ ]:
PRODUCT_REPO = Path("/content/kandinsky-v2a-colab")
if not (PRODUCT_REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/AytugSahinkanat/kandinsky-v2a-colab.git",
                    str(PRODUCT_REPO)], check=True)

manifest = {
    "schema": "kandinsky-v2a-review-bundle-v2",
    "study_title": "Kandinsky Video-to-Audio Six-Scene Pilot",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": {
        "name": "Kandinsky 6.0",
        "checkpoint": "lite-distill",
        "checkpoint_repo": "kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers",
        "checkpoint_revision": CHECKPOINT_REVISION,
    },
    "source_repo": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": SOURCE_COMMIT,
    "generation": {
        "duration_seconds": GENERATION_SECONDS,
        "seed": SEED,
        "gpu": gpu.stdout.strip(),
        "prompt_expansion": "disabled",
        "attention_engine": "sdpa",
    },
    "sources": SOURCE_METADATA,
    "scenes": successful_scenes,
    "candidates": candidates,
}
BUNDLE.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    for item in SCENES:
        archive.write(item["source_path_local"], "source_originals/" + item["filename"])
    for scene in successful_scenes:
        source_item = next(x for x in SCENES if x["scene_id"] == scene["scene_id"])
        archive.write(source_item["scene_video_local"], scene["video_path"])
    for candidate in candidates:
        archive.write(GEN_DIR / (candidate["candidate_id"] + ".wav"), candidate["audio_path"])
        archive.write(GEN_DIR / (candidate["candidate_id"] + ".mp4"), candidate["generated_mp4_path"])
print("Research bundle:", BUNDLE, "(", round(BUNDLE.stat().st_size / 1e6, 1), "MB )")

STUDY_DIR = WORK / ("participant_study_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
builder = PRODUCT_REPO / "tools/build_kandinsky_review.py"
if not builder.is_file():
    raise FileNotFoundError("Review builder is missing from the public repository: " + str(builder))
result = subprocess.run([sys.executable, str(builder), str(BUNDLE), str(STUDY_DIR)],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr[-5000:])
    raise RuntimeError("Could not create the listener package.")
build_result = json.loads(result.stdout)
participant_zip = Path(build_result["participant_zip"])
researcher_key = Path(build_result["researcher_key"])
print("Participant ZIP:", participant_zip)
print("Private researcher key:", researcher_key)
print("Keep researcher_key.json private; send participants only Participant_Package.zip.")


## 6 — Download the rating materials

The researcher ZIP is for you. Extract it and send only Participant_Package.zip to raters. The package works offline after extraction; each rater downloads their JSON response and sends it back to you. There is no automatic upload or public hosting.

In [ ]:
from google.colab import files

DOWNLOAD = WORK / "Kandinsky_Six_Scene_Review_Artifacts.zip"
with zipfile.ZipFile(DOWNLOAD, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(participant_zip, "Participant_Package.zip")
    archive.write(researcher_key, "researcher_key.json")
    archive.write(BUNDLE, "generation_bundle.zip")
print("Final artifact:", DOWNLOAD, "(", round(DOWNLOAD.stat().st_size / 1e6, 1), "MB )")
files.download(str(DOWNLOAD))


## What the ratings will tell us

Rate **scene fit**, **audio-video timing**, **sound quality**, **noise/artifacts**, unwanted music or speech, and whether you would keep the audio as-is, after editing, or not at all.

This small set is for finding strong and weak event families. It does not compare different seeds, prove a general success rate, or isolate the effect of the prompt from the video. Kandinsky receives both the short prompt and the video for every scene.
